In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # project root

In [ ]:
import rasterio
from pathlib import Path
import numpy as np

from spm.config.config import ModelConfig
from spm.models.yolo import YOLOModel
from spm.utils.profiling import size_it, time_it
from spm.visualization.overlays import visualize
from spm.preprocessing.image_processing import binary_mask_to_contours

from sahi.postprocess.combine import GreedyNMMPostprocess, NMMPostprocess
from sahi.postprocess.backends import set_postprocess_backend

from spatial_mask_merging.smm.predictions import SMMPrediction
from spatial_mask_merging.smm.smm import SpatialMaskMerger

from utils.adapters import PredictionAdapter, SAHIPrediction

## Run Full Inference Pipeline

In [ ]:
model_path = "../runs/segment/yolo-seg-whu/weights/best.pt"
tiff_path = Path("../whole_cropped_2500m.tif")

In [ ]:
# Inference configuration
tile_size = 1500
batch_size = 4
overlap = 0.2
device = "cuda"  # or "cpu"

In [ ]:
config = ModelConfig(
    model_path=model_path,
    tile_size=tile_size,
    batch_size=batch_size,
    overlap=overlap,
    device=device,
    )

In [ ]:
model = YOLOModel(config)

In [ ]:
prediction, unmerged_prediction = model(
                                    tiff_path,
                                    vector_format="gpkg",
                                    show_viz=True,
                                    merge_only_border=False,
                                    get_seg_from_binary_mask=True
                                    )

In [ ]:
len(unmerged_prediction.polygons)

In [ ]:
# Adapt the unmerged prediction to support both SAHI and SMM formats
prediction_adapter = PredictionAdapter(unmerged_prediction)
unmerged_sahi_predictions = prediction_adapter.sahi
unmerged_smm_predictions = prediction_adapter.smm

## Merge Predictions with SAHI(GreedyNMM)

In [ ]:
postprocess = GreedyNMMPostprocess(
        match_threshold=0.1,
        match_metric="IOS",
        class_agnostic=False,
    )

In [ ]:
# Set SAHI backend to numpy to process predictions on CPU
set_postprocess_backend("numpy")

In [ ]:
@size_it
@time_it
def _postprocess(sahi_predictions):
    merged_sahi_predictions = postprocess(sahi_predictions.predictions)
    return merged_sahi_predictions

In [ ]:
# Postprocess the unmerged SAHI predictions to merge overlapping polygons
merged_sahi_predictions = _postprocess(unmerged_sahi_predictions)

In [ ]:
# Convert merged SAHI predictions back to SPM format
sahi_prediction = SAHIPrediction()
sahi_prediction.predictions = merged_sahi_predictions
sahi_merged_predictions_spm = PredictionAdapter(sahi_prediction).spm

In [ ]:
# Visualize the GNMM merged predictions on the original TIFF image
viz_dir = Path("predictions/viz")
viz_dir.mkdir(parents=True, exist_ok=True)
output_path = viz_dir / f"{tiff_path.stem}_sahi_gnmm_prediction.png"
with rasterio.open(tiff_path) as src:
    visualize(src, sahi_merged_predictions_spm, output_path)

## Merege Predictions with SMM

In [ ]:
@size_it
@time_it
def smm_merger(prediction: SMMPrediction, image_size_hw: tuple[int, int]):
    # Initialize SMM with paper parameters
    merger = SpatialMaskMerger(
        tau_d=5.0,      # Distance threshold (pixels)
        tau_i=0.5,       # IoU threshold
        rho=10.0,        # R-tree search radius (pixels)
        beta1=0.3,       # Distance weight
        beta2=0.5,       # IoU weight
        beta3=0.2,       # Confidence weight
        gamma=0.5,       # Anti-chaining threshold
        lambda_=1.0      # Clustering penalty
    )

    return merger.merge(prediction, image_size_hw=image_size_hw)

In [ ]:
smm_merged_obj = smm_merger(unmerged_smm_predictions, image_size_hw=unmerged_prediction.image_shape)

In [ ]:
# Convert SMM obj to SMMPrediction
smm_merged_predictions = SMMPrediction(image_name=tiff_path)

for pred in smm_merged_obj:

    contour = binary_mask_to_contours(pred["mask"].astype(np.uint8), normalize=False)
    segmentation = np.array(contour).reshape(1, -1, 2).tolist()

    smm_merged_predictions.add_annotation(
        type=pred["label"],
        class_id=pred["label"],
        confidence=pred["score"],
        segmentation=segmentation,
        bbox=pred["bbox"]
    )

In [ ]:
# Convert merged SMM predictions back to SPM format
smm_merged_predictions_to_spm = PredictionAdapter(smm_merged_predictions).spm

In [ ]:
# Visualize the SMM merged predictions on the original TIFF image
viz_dir = Path("predictions/viz")
viz_dir.mkdir(parents=True, exist_ok=True)
output_path = viz_dir / f"{tiff_path.stem}_smm_prediction.png"
with rasterio.open(tiff_path) as src:
    visualize(src, smm_merged_predictions_to_spm, output_path)